# Cloud workload forecasting — analysis notebook

Reproduces every number and figure in `reports/final_report.md`.
Run from the repository root after `pip install -e ".[dev,analysis]"` and extracting `data-v1` into `challenge_data/`.

**Holdout protocol.** Sections 1–6 are exploratory and use the full 134-slot series (that is what "understand the workload" means; nothing here feeds model selection — the selection rule was fixed in code before looking at the holdout). Section 7 runs the production pipeline, which only sees the first 110 slots. Holdout targets are read **only** in Section 8, after predictions are produced.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd() if (pathlib.Path.cwd() / "src").exists() else pathlib.Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
import numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from cwfc.data import load_dataset, split_holdout, vm_matrix
from cwfc import metrics as M, policy as P
from cwfc.baselines import phase_profile
DATA, FIG = ROOT / "challenge_data", ROOT / "reports" / "figures"
FIG.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .3, "axes.spines.top": False, "axes.spines.right": False})
def save(fig, name): fig.tight_layout(); fig.savefig(FIG / f"{name}.png", dpi=130); plt.close(fig)
pd.set_option("display.width", 160)

## 1. Data quality, cadence and coverage

In [2]:
ds = load_dataset(DATA)
r, meta, tg = ds.readings, ds.metadata, ds.targets
print(pd.Series(ds.report.as_dict()))
ts = np.sort(r.timestamp.unique()); gaps = np.diff(ts)
per_vm = r.groupby("vm_id").timestamp.agg(["count", "min", "max"])
print("timestamps:", len(ts), "range", ts.min(), "-", ts.max(), "s; unique gaps:", np.unique(gaps))
print("readings per VM:", per_vm["count"].unique(), "| first/last slot per VM:", per_vm["min"].unique(), per_vm["max"].unique())
print("VMs per timestamp:", tg.active_vm_count.unique())
print("min<=avg<=max violations:", int(((r.min_cpu > r.avg_cpu + 1e-9) | (r.avg_cpu > r.max_cpu + 1e-9)).sum()))
print("metadata rows:", len(meta), "unique vm_id:", meta.vm_id.nunique(), "| CPU VMs without metadata:", len(set(r.vm_id) - set(meta.vm_id)))

rows_raw                 1340000
rows_invalid                   0
rows_duplicate                 0
n_vms                      10000
n_timestamps_observed        134
n_grid_slots                 134
n_empty_grid_slots             0
off_grid_timestamps            0
notes                         []
dtype: object


timestamps: 134 range 0 - 39900 s; unique gaps: [300]
readings per VM: [134] | first/last slot per VM: [0] [39900]
VMs per timestamp: [10000]
min<=avg<=max violations: 0


metadata rows: 10000 unique vm_id: 10000 | CPU VMs without metadata: 0


### VM lifetimes and churn

In [3]:
m = meta.set_index("vm_id")
print("created:", m.timestamp_vm_created.describe()[["min", "max"]].to_dict())
deleted_in_trace = m.timestamp_vm_deleted < m.timestamp_vm_deleted.max()
print(f"VMs deleted before the end of the 30-day trace: {deleted_in_trace.mean():.1%}")
print("remaining lifetime of those (days):", (m.timestamp_vm_deleted[deleted_in_trace] / 86400).describe().round(1).to_dict())
print("deleted inside the 11.1 h window:", int((m.timestamp_vm_deleted <= ts.max()).sum()))
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(m.timestamp_vm_deleted / 86400, bins=60, color="#4c72b0")
ax.set(xlabel="deletion time (days since trace start)", ylabel="VMs", title="All VMs alive for the whole window; 18% are deleted later in the month")
save(fig, "lifetimes")

created: {'min': 0.0, 'max': 0.0}
VMs deleted before the end of the 30-day trace: 17.9%
remaining lifetime of those (days): {'count': 1787.0, 'mean': 10.4, 'std': 7.9, 'min': 0.5, '25%': 4.1, '50%': 8.5, '75%': 16.1, 'max': 30.0}
deleted inside the 11.1 h window: 0


## 2. CPU utilisation distributions

In [4]:
W = vm_matrix(r, ts)                      # VM x time avg_cpu
MX = vm_matrix(r, ts, "max_cpu")
vm_mean = W.mean(axis=1)
print("reading-level avg_cpu percentiles:", np.percentile(r.avg_cpu, [5, 25, 50, 75, 95, 99]).round(2))
print("per-VM mean percentiles:", np.percentile(vm_mean, [5, 25, 50, 75, 95, 99]).round(2))
print(f"VMs with mean <5%: {(vm_mean < 5).mean():.1%}; >50%: {(vm_mean > 50).mean():.1%}")
srt = np.sort(vm_mean.to_numpy())[::-1]; share = np.cumsum(srt) / srt.sum()
print(f"top 1% of VMs -> {share[99]:.1%} of CPU; top 10% -> {share[999]:.1%}")
cv = W.std(axis=1) / W.mean(axis=1); agg = W.mean(axis=0)
print(f"median per-VM CV over the window: {cv.median():.2f}; aggregate CV: {agg.std() / agg.mean():.3f}")
hot = (MX > 80).mean(axis=0)
print(f"share of VMs with a 5-min max_cpu >80%: mean {hot.mean():.1%}, max {hot.max():.1%}")
print("corr(metadata avg_cpu, window mean) =", round(np.corrcoef(m.loc[W.index, "avg_cpu"], vm_mean)[0, 1], 3))
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].hist(r.avg_cpu, bins=np.logspace(-2, 2, 60), color="#4c72b0"); ax[0].set_xscale("log")
ax[0].set(xlabel="avg_cpu (%) — log scale", ylabel="readings", title="5-min readings: heavily right-skewed")
ax[1].plot(np.arange(1, len(share) + 1) / len(share) * 100, share * 100, color="#c44e52")
ax[1].set(xlabel="% of VMs (busiest first)", ylabel="% of total CPU", title="CPU concentration"); ax[1].set_xscale("log")
save(fig, "distributions")

reading-level avg_cpu percentiles: [ 0.77  2.48  5.11  9.62 37.13 87.35]
per-VM mean percentiles: [ 0.93  2.87  5.48  9.93 33.28 78.37]
VMs with mean <5%: 46.4%; >50%: 2.2%
top 1% of VMs -> 9.6% of CPU; top 10% -> 43.6%
median per-VM CV over the window: 0.29; aggregate CV: 0.021
share of VMs with a 5-min max_cpu >80%: mean 2.9%, max 4.5%
corr(metadata avg_cpu, window mean) = 0.871


## 3. Aggregate targets and temporal structure

In [5]:
sp = split_holdout(tg, 24)
y = tg.set_index("timestamp").aggregate_cpu_utilization
hours = (y.index - y.index[0]) / 3600
fig, ax = plt.subplots(figsize=(10, 3.4))
ax.plot(hours, y.values, color="#4c72b0", lw=1.2, label="aggregate_cpu_utilization")
ax.plot(hours, y.rolling(12, center=True, min_periods=6).mean(), color="k", lw=1, ls="--", label="centred 1 h mean")
ax.axvspan(hours[110], hours[-1], color="orange", alpha=.15, label="holdout (24 slots)")
ax.axvspan(hours[55], hours[70], color="red", alpha=.07, label="batch surge")
ax.set(xlabel="hours since trace start", ylabel="mean avg_cpu (%)", title="Aggregate CPU: flat level ~9.4%, an hourly cycle, and one 1.3 h surge")
ax.legend(loc="upper right", fontsize=8, ncol=2); save(fig, "aggregate_series")
print("active_vm_count unique values:", tg.active_vm_count.unique())
print(y.describe().round(3).to_dict())

active_vm_count unique values: [10000]
{'count': 134.0, 'mean': 9.388, 'std': 0.195, 'min': 8.984, '25%': 9.251, '50%': 9.367, '75%': 9.486, 'max': 9.912}


In [6]:
d = np.diff(y.values)
acf = [np.corrcoef(d[:-k], d[k:])[0, 1] for k in range(1, 37)]
prof_full = phase_profile(y.values[:110])      # profile as the model sees it (training only)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].bar(range(1, 37), acf, color=["#c44e52" if k % 12 == 0 else "#4c72b0" for k in range(1, 37)])
ax[0].set(xlabel="lag (5-min slots)", ylabel="ACF of first differences", title="Hourly (lag 12/24/36) recurrence")
ax[1].bar(np.arange(12) * 5, prof_full, width=4, color="#55a868")
ax[1].set(xlabel="minute within trace-hour", ylabel="deviation from hourly mean (pp)", title="Within-hour profile (training window)")
save(fig, "seasonality")
print("ACF lags 1,12,24:", np.round([acf[0], acf[11], acf[23]], 3)); print("profile:", prof_full.round(3))

ACF lags 1,12,24: [-0.401  0.708  0.752]
profile: [-0.018  0.011  0.022 -0.063 -0.172  0.116 -0.037 -0.048  0.063  0.081
  0.01   0.035]


### Who drives the hourly cycle and the surge?

In [7]:
ph = np.arange(len(ts)) % 12
Wv = W.to_numpy()
dev = Wv - pd.DataFrame(Wv).T.rolling(12, center=True, min_periods=6).mean().T.to_numpy()
amp = pd.Series(np.nanmean(dev[:, ph == 5], 1) - np.nanmean(dev[:, ph == 4], 1), index=W.index)
s = amp.sort_values(ascending=False)
print(f"hourly swing: top 1% VMs = {s[:100].sum() / amp.sum():.0%} of the aggregate swing, top 5% = {s[:500].sum() / amp.sum():.0%}")
surge = W.iloc[:, 55:71].mean(axis=1) - pd.concat([W.iloc[:, 40:54], W.iloc[:, 72:86]], axis=1).mean(axis=1)
ss = surge.sort_values(ascending=False)
print(f"surge: +{surge.mean():.2f} pp aggregate; top 50 VMs = {ss[:50].sum() / surge.sum():.0%}, top 100 = {ss[:100].sum() / surge.sum():.0%}")
top = m.loc[ss.index[:100]]
print("top-100 surge VMs: deployments", top.deployment_id.nunique(), "| subscriptions", top.subscription_id.nunique())
print((surge.groupby(m.loc[W.index, "vm_category"]).sum() / surge.sum()).round(2).to_dict())

hourly swing: top 1% VMs = 42% of the aggregate swing, top 5% = 105%
surge: +0.39 pp aggregate; top 50 VMs = 40%, top 100 = 62%
top-100 surge VMs: deployments 41 | subscriptions 41
{'Delay-insensitive': 0.79, 'Interactive': 0.18, 'Unknown': 0.03}


## 4. Segments

In [8]:
seg = m.loc[W.index]
rows = []
for col in ["vm_category", "virtual_core_bucket", "memory_gb_bucket"]:
    for k, idx in seg.groupby(col).groups.items():
        sub = W.loc[idx]; a = sub.mean(axis=0)
        dd = np.diff(a.values)
        rows.append({"segment": col, "value": k, "vms": len(idx), "mean_cpu": a.mean(), "p95_vm_mean": np.percentile(sub.mean(axis=1), 95),
                     "agg_cv": a.std() / a.mean(), "hourly_acf12": np.corrcoef(dd[:-12], dd[12:])[0, 1],
                     "hot_share": (MX.loc[idx] > 80).mean().mean()})
segtab = pd.DataFrame(rows).round(3); print(segtab.to_string(index=False))
fig, ax = plt.subplots(figsize=(10, 3.2))
for k, idx in seg.groupby("vm_category").groups.items():
    a = W.loc[idx].mean(axis=0); ax.plot(hours, a / a.iloc[:110].mean(), label=f"{k} (n={len(idx)})", lw=1)
ax.axvline(hours[110], color="orange"); ax.set(xlabel="hours", ylabel="CPU / training mean", title="Segment series (normalised)")
ax.legend(fontsize=8); save(fig, "segments")
cores = seg.virtual_core_bucket.astype(str).map({"2": 2, "4": 4, "8": 8, "24": 24, ">24": 32})
cw = W.mul(cores, axis=0).sum() / cores.sum()
print(f"core-weighted utilisation: mean {cw.mean():.2f}% (unweighted {agg.mean():.2f}%); busy vCPUs ≈ {cw.mean() / 100 * cores.sum():.0f} of {cores.sum()} (>24 bucket assumed 32)")

            segment             value  vms  mean_cpu  p95_vm_mean  agg_cv  hourly_acf12  hot_share
        vm_category Delay-insensitive 6180     8.236       25.747   0.025         0.732      0.022
        vm_category       Interactive 3091    11.493       46.025   0.023         0.470      0.040
        vm_category           Unknown  729    10.233       48.393   0.063         0.491      0.042
virtual_core_bucket                 2 7534     9.725       34.145   0.025         0.621      0.030
virtual_core_bucket                24   62    11.397       50.590   0.034         0.074      0.009
virtual_core_bucket                 4 1908     7.928       31.343   0.028         0.538      0.025
virtual_core_bucket                 8  486     9.700       35.657   0.043         0.594      0.022
virtual_core_bucket               >24   10     6.618       19.065   0.300         0.835      0.157
   memory_gb_bucket                 2 3904     8.775       15.885   0.024         0.563      0.019
   memory_

core-weighted utilisation: mean 9.29% (unweighted 9.39%); busy vCPUs ≈ 2638 of 28396 (>24 bucket assumed 32)


## 5. Peak and low-demand periods

In [9]:
thr = y.quantile(0.9)
pk = np.where(y.values >= thr)[0]
print(f"top-decile threshold {thr:.2f}%; peak slots: {pk.tolist()}")
print("min", y.idxmin(), round(y.min(), 3), "| max", y.idxmax(), round(y.max(), 3))
print("peak-to-trough of 1 h mean:", round(y.rolling(12).mean().max() - y.rolling(12).mean().min(), 3), "pp")

top-decile threshold 9.66%; peak slots: [55, 56, 57, 58, 59, 60, 61, 62, 63, 65, 66, 67, 68, 69]
min 33300 8.984 | max 16500 9.912
peak-to-trough of 1 h mean: 0.655 pp


## 6. Anomaly scan (one-step-ahead residuals, training-window only)

In [10]:
from cwfc.baselines import ses_profile_steps
ytr = sp.train.aggregate_cpu_utilization.values
one = np.array([ses_profile_steps(ytr[:o], np.array([1]))[0] for o in range(24, 110)])
res1 = ytr[24:110] - one
z = (res1 - np.median(res1)) / (1.4826 * np.median(np.abs(res1 - np.median(res1))))
flag = np.where(np.abs(z) > 3)[0] + 24
print("anomalous slots (|robust z|>3):", flag.tolist(), "z:", np.round(z[np.abs(z) > 3], 1).tolist())
fig, ax = plt.subplots(figsize=(10, 2.6))
ax.bar(np.arange(24, 110), z, color=np.where(np.abs(z) > 3, "#c44e52", "#8da0cb"))
ax.axhline(3, ls="--", c="k", lw=.8); ax.axhline(-3, ls="--", c="k", lw=.8)
ax.set(xlabel="slot", ylabel="robust z", title="One-step residuals: surge onset is the only sustained anomaly"); save(fig, "anomalies")

anomalous slots (|robust z|>3): [54, 55, 92] z: [3.1, 6.9, 3.1]


## 7. Forecasting pipeline (training window only)
This is exactly what `python -m cwfc.predict` runs.

In [11]:
from cwfc.predict import forecast_all
res = forecast_all(DATA, horizon=24, n_folds=16)
summ = res["summaries"]["aggregate_cpu_utilization"]; err = res["errors"]["aggregate_cpu_utilization"]
summ.round(4)

[active_vm_count] selected persistence: lowest mean backtest MAE over 16 rolling origins
            model  mae  rmse  wape_pct  mae_vs_persistence  peak_mae
moving_average_12  0.0   0.0       0.0                 0.0       0.0
      persistence  0.0   0.0       0.0                 0.0       0.0
         drift_24  0.0   0.0       0.0                 inf       0.0


[aggregate_cpu_utilization] selected ses_profile: lowest mean backtest MAE over 16 rolling origins
            model    mae   rmse  wape_pct  mae_vs_persistence  peak_mae
      ses_profile 0.1976 0.2508    2.1040              0.8499    0.0679
 ensemble_gbm_ses 0.2103 0.2532    2.2389              0.9044    0.1102
 moving_average_6 0.2310 0.2827    2.4590              0.9933    0.1486
      persistence 0.2326 0.2861    2.4755              1.0000    0.1006
        panel_gbm 0.2381 0.2786    2.5348              1.0239    0.2022
 seasonal_profile 0.2474 0.2837    2.6332              1.0637    0.2221
moving_average_12 0.2518 0.2948    2.6806              1.0829    0.2393
moving_average_24 0.2683 0.2976    2.8557              1.1536    0.3269
seasonal_naive_1h 0.2693 0.3171    2.8664              1.1579    0.3092
      panel_ridge 0.3001 0.3504    3.1949              1.2906    0.1365
         drift_24 0.4377 0.5422    4.6594              1.8822    0.1606
done in 210.9s


,model,baseline,folds,mae,rmse,wape_pct,bias,mae_vs_persistence,mae_fold_std,worst_fold_mae,peak_mae
0,ses_profile,False,13,0.1976,0.2508,2.1040,-0.1866,0.8499,0.1025,0.3599,0.0679
1,ensemble_gbm_ses,False,13,0.2103,0.2532,2.2389,-0.1857,0.9044,0.0949,0.3691,0.1102
2,moving_average_6,True,13,0.2310,0.2827,2.4590,-0.1932,0.9933,0.1022,0.3718,0.1486
3,persistence,True,13,0.2326,0.2861,2.4755,-0.2053,1.0000,0.0967,0.4638,0.1006
4,panel_gbm,False,13,0.2381,0.2786,2.5348,-0.1849,1.0239,0.0959,0.4022,0.2022
5,seasonal_profile,False,13,0.2474,0.2837,2.6332,-0.1901,1.0637,0.0987,0.3872,0.2221
6,moving_average_12,True,13,0.2518,0.2948,2.6806,-0.1901,1.0829,0.0958,0.3900,0.2393
7,moving_average_24,True,13,0.2683,0.2976,2.8557,-0.1762,1.1536,0.0623,0.3535,0.3269
8,seasonal_naive_1h,True,13,0.2693,0.3171,2.8664,-0.1901,1.1579,0.0946,0.3840,0.3092
9,panel_ridge,False,13,0.3001,0.3504,3.1949,-0.2728,1.2906,0.1030,0.4660,0.1365


In [12]:
err["ae"] = err.error.abs()
n_models = err.model.nunique(); per_origin = err.groupby("origin").model.nunique()
common = per_origin[per_origin == n_models].index      # folds on which every candidate could run
errc = err[err.origin.isin(common)]
show = ["persistence", "moving_average_6", "seasonal_naive_1h", "ses_profile", "panel_gbm", "ensemble_gbm_ses"]
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
for name in show:
    g = errc[errc.model == name]
    ax[0].plot(g.groupby("step").ae.mean(), label=name, lw=2 if name == "ses_profile" else 1)
    ax[1].plot(err[err.model == name].groupby("origin").ae.mean(), marker="o", ms=3, label=name, lw=2 if name == "ses_profile" else 1)
ax[0].set(xlabel="forecast step (5 min)", ylabel="MAE (pp)", title=f"Backtest MAE by step ({len(common)} common folds)")
ax[1].set(xlabel="forecast origin (slots of history)", ylabel="MAE (pp)", title="Backtest MAE by origin")
ax[1].axvspan(32, 70, color="red", alpha=.06, label="window overlaps surge"); ax[0].legend(fontsize=7); save(fig, "backtest")
pv = err.pivot_table(index="model", columns="origin", values="ae").loc[show]
wins = (pv.loc["ses_profile"] < pv.loc["persistence"]).mean()
print("MAE by step bucket (common folds):"); print(errc[errc.model.isin(show)].assign(bucket=lambda d: pd.cut(d.step, [0, 6, 12, 24], labels=["1-6", "7-12", "13-24"])).pivot_table(index="model", columns="bucket", values="ae", observed=True).round(3))
print(f"ses_profile beats persistence on {wins:.0%} of origins"); pv.round(3)

MAE by step bucket (common folds):
bucket               1-6   7-12  13-24
model                                 
ensemble_gbm_ses   0.132  0.177  0.266
moving_average_6   0.152  0.202  0.285
panel_gbm          0.189  0.213  0.275
persistence        0.136  0.198  0.298
seasonal_naive_1h  0.271  0.210  0.298
ses_profile        0.094  0.162  0.267
ses_profile beats persistence on 62% of origins


origin,48,51,53,56,58,61,63,66,68,71,73,76,78,81,83,86
model,,,,,,,,,,,,,,,,
persistence,0.298,0.233,0.416,0.296,0.244,0.207,0.302,0.464,0.301,0.220,0.257,0.110,0.257,0.131,0.102,0.132
moving_average_6,0.333,0.299,0.324,0.198,0.190,0.311,0.320,0.328,0.353,0.372,0.330,0.158,0.096,0.085,0.124,0.140
seasonal_naive_1h,0.309,0.312,0.321,0.294,0.266,0.264,0.300,0.361,0.384,0.384,0.378,0.280,0.224,0.143,0.121,0.102
ses_profile,0.320,0.299,0.298,0.232,0.204,0.238,0.280,0.360,0.336,0.243,0.278,0.044,0.114,0.064,0.070,0.106
panel_gbm,NaN,NaN,NaN,0.185,0.158,0.183,0.216,0.374,0.402,0.347,0.367,0.217,0.226,0.172,0.155,0.093
ensemble_gbm_ses,NaN,NaN,NaN,0.156,0.168,0.203,0.242,0.367,0.369,0.294,0.323,0.123,0.170,0.114,0.109,0.095


In [13]:
# Is the panel model learning anything at VM level? Compare VM-level vs aggregate skill on the last fold.
from cwfc.panel import PanelForecaster
from cwfc.baselines import persistence_steps
hist = res["history"]; o = 86; steps = np.arange(1, 25)
sub = hist.upto(o)
pf = PanelForecaster(learner="gbm")
agg_p, vm_p = pf.forecast(sub.X, sub.static, steps, return_vm=True)
act_vm = hist.X[:, o:o + 24]
pers_vm = np.repeat(sub.X[:, -1:], 24, axis=1)
print("VM-level MAE   panel %.3f vs persistence %.3f" % (np.nanmean(np.abs(act_vm - vm_p)), np.nanmean(np.abs(act_vm - pers_vm))))
a = hist.cpu[o:o + 24]
print("aggregate MAE  panel %.3f vs persistence %.3f" % (M.mae(a, agg_p), M.mae(a, pers_vm.mean(0))))
print("training rows:", pf.n_train_rows_)

VM-level MAE   panel 1.974 vs persistence 2.360
aggregate MAE  panel 0.093 vs persistence 0.132
training rows: 1530000


## 8. Holdout evaluation (run once, after predictions were written)
All candidates are scored for transparency only; the submitted model was selected in Section 7 from backtests.

In [14]:
from cwfc.models import models_for
pred = res["predictions"]; ivl = res["intervals"]
hold = sp.holdout.set_index("timestamp").loc[pred.timestamp]
act = hold.aggregate_cpu_utilization.to_numpy()
steps = np.arange(1, 25)
hold_rows = []
for mdl in models_for("aggregate_cpu_utilization"):
    p = np.clip(mdl.fn(hist, steps), 0, 100)
    pers = persistence_steps(hist.cpu, steps)
    hi = act >= np.quantile(act, .75)
    hold_rows.append({"model": mdl.name, "mae": M.mae(act, p), "rmse": M.rmse(act, p), "wape_pct": M.wape(act, p),
                      "bias": float(np.mean(act - p)), "mae_vs_persistence": M.normalized_mae(act, p, pers), "peak_mae": M.mae(act[hi], p[hi])})
holdtab = pd.DataFrame(hold_rows).sort_values("mae").round(4)
print("active_vm_count holdout MAE:", M.mae(hold.active_vm_count, pred.active_vm_count))
cov80 = M.interval_coverage(act, ivl.aggregate_cpu_utilization_q10, ivl.aggregate_cpu_utilization_q90)
cov90 = M.interval_coverage(act, ivl.aggregate_cpu_utilization_q05, ivl.aggregate_cpu_utilization_q95)
print(f"interval coverage on holdout: 80% band {cov80:.0%}, 90% band {cov90:.0%}")
holdtab

active_vm_count holdout MAE: 0.0
interval coverage on holdout: 80% band 100%, 90% band 100%


,model,mae,rmse,wape_pct,bias,mae_vs_persistence,peak_mae
6,seasonal_profile,0.1149,0.1394,1.2395,0.0526,0.7827,0.2087
3,moving_average_24,0.1169,0.1537,1.2612,0.0195,0.7964,0.2235
2,moving_average_12,0.1230,0.1613,1.3271,0.0526,0.8380,0.2567
8,panel_ridge,0.1310,0.1590,1.4138,0.0231,0.8928,0.2281
1,moving_average_6,0.1336,0.1713,1.4410,0.0781,0.9099,0.2822
7,ses_profile,0.1396,0.1756,1.5066,0.1190,0.9514,0.2751
4,seasonal_naive_1h,0.1400,0.1590,1.5109,0.0526,0.9541,0.2283
0,persistence,0.1468,0.1848,1.5836,0.1044,1.0000,0.3085
10,ensemble_gbm_ses,0.1474,0.1820,1.5901,0.0968,1.0041,0.2931
9,panel_gbm,0.1649,0.1981,1.7789,0.0745,1.1233,0.3111


In [15]:
fig, ax = plt.subplots(figsize=(10, 3.4))
hh = hours[:110]; ax.plot(hh[-36:], ytr[-36:], color="#4c72b0", lw=1.2, label="training actual")
hx = hours[110:]
ax.plot(hx, act, color="k", lw=1.4, label="holdout actual")
ax.plot(hx, pred.aggregate_cpu_utilization, color="#c44e52", lw=1.6, label="forecast (ses_profile)")
ax.plot(hx, persistence_steps(hist.cpu, steps), color="grey", ls=":", label="persistence")
ax.fill_between(hx, ivl.aggregate_cpu_utilization_q05, ivl.aggregate_cpu_utilization_q95, color="#c44e52", alpha=.12, label="90% band")
ax.fill_between(hx, ivl.aggregate_cpu_utilization_q10, ivl.aggregate_cpu_utilization_q90, color="#c44e52", alpha=.18, label="80% band")
ax.set(xlabel="hours since trace start", ylabel="aggregate CPU (%)", title="Two-hour holdout forecast")
ax.legend(fontsize=7, ncol=3, loc="lower left"); save(fig, "holdout_forecast")
e = act - pred.aggregate_cpu_utilization.to_numpy()
print("holdout error by hour: first hour MAE %.3f bias %+.3f | second hour MAE %.3f bias %+.3f" % (np.abs(e[:12]).mean(), e[:12].mean(), np.abs(e[12:]).mean(), e[12:].mean()))
print("largest misses (slot, actual, forecast):", sorted(zip(np.abs(e).round(3), (110 + np.arange(24)).tolist(), act.round(2)), reverse=True)[:3])

holdout error by hour: first hour MAE 0.059 bias +0.017 | second hour MAE 0.221 bias +0.221
largest misses (slot, actual, forecast): [(np.float64(0.37), 125, np.float64(9.64)), (np.float64(0.315), 131, np.float64(9.5)), (np.float64(0.291), 132, np.float64(9.42))]


## 9. Failure analysis

In [16]:
g = err[err.model == "ses_profile"].copy()
g["phase"] = ((g.timestamp - ts[0]) // 300) % 12
g["surge_window"] = g.timestamp.between(ts[55], ts[70])
print("MAE inside vs outside surge slots:", g.groupby("surge_window").ae.mean().round(3).to_dict())
print("share of squared error that is fold-level bias:",
      round((g.groupby("origin").error.mean() ** 2).mean() / (g.error ** 2).mean(), 2))
print("MAE by within-hour phase:", g.groupby("phase").ae.mean().round(3).to_dict())
thr_bt = g.actual.quantile(.9)
print("peak (top-decile actual) MAE %.3f, bias %+.3f vs overall MAE %.3f" % (g[g.actual >= thr_bt].ae.mean(), g[g.actual >= thr_bt].error.mean(), g.ae.mean()))

MAE inside vs outside surge slots: {False: 0.211, True: 0.236}
share of squared error that is fold-level bias: 0.73
MAE by within-hour phase: {0: 0.168, 1: 0.201, 2: 0.216, 3: 0.24, 4: 0.201, 5: 0.194, 6: 0.235, 7: 0.309, 8: 0.204, 9: 0.208, 10: 0.243, 11: 0.194}
peak (top-decile actual) MAE 0.291, bias +0.287 vs overall MAE 0.218


## 10. Capacity policy

In [17]:
taus = [0.5, 0.8, 0.9, 0.95, 0.99]
qt = P.residual_quantile_table(err, "ses_profile", taus, 24)
fc = pred.aggregate_cpu_utilization.to_numpy()
train_max = np.full(24, np.nanmax(ytr))
extra = {"static: training max": train_max,
         "persistence + 5% relative margin": persistence_steps(hist.cpu, steps) * 1.05,
         "forecast + 5% relative margin": fc * 1.05}
for cu, co, label in [(1, 1, "symmetric cost"), (9, 1, "shortfall 9x worse (tau*=0.9)")]:
    print(f"--- holdout, {label} ---")
    print(P.policy_table(act, fc, steps, qt, cu, co, extra).round(3).to_string(index=False))
# Same evaluation on the backtest folds (in-sample for the quantiles, so optimistic)
bt_rows = []
for tau in taus:
    prov = g.pred + qt.loc[g.step, tau].to_numpy()
    bt_rows.append({"policy": f"forecast + Q{tau}", **P.evaluate(g.actual, prov, 9, 1)})
for pct in [0.02, 0.04, 0.05]:
    bt_rows.append({"policy": f"forecast + {pct:.0%} relative", **P.evaluate(g.actual, g.pred * (1 + pct), 9, 1)})
pg = err[err.model == "persistence"]
bt_rows.append({"policy": "persistence + 5% relative", **P.evaluate(pg.actual, pg.pred * 1.05, 9, 1)})
print("--- backtest folds, shortfall 9x worse (quantiles calibrated on these same folds, so optimistic) ---"); print(pd.DataFrame(bt_rows).round(3).to_string(index=False))
print("headroom at tau=0.9 by step (pp):", qt[0.9].iloc[[0, 5, 11, 17, 23]].round(3).to_dict())

--- holdout, symmetric cost ---
                          policy  shortfall_rate  mean_shortfall  max_shortfall  mean_headroom  cost
                forecast + Q0.50           0.875           0.252          0.556          0.004 0.256
                forecast + Q0.80           0.583           0.083          0.303          0.031 0.115
                forecast + Q0.90           0.000           0.000          0.000          0.218 0.218
                forecast + Q0.95           0.000           0.000          0.000          0.336 0.336
                forecast + Q0.99           0.000           0.000          0.000          0.484 0.484
            static: training max           0.000           0.000          0.000          0.643 0.643
persistence + 5% relative margin           0.042           0.001          0.013          0.354 0.355
   forecast + 5% relative margin           0.000           0.000          0.000          0.338 0.338
--- holdout, shortfall 9x worse (tau*=0.9) ---
            

In [18]:
# In absolute capacity terms (vCPUs busy, core-weighted, >24 bucket assumed = 32 cores)
total_cores = int(cores.sum())
for tau in [0.5, 0.9, 0.95]:
    hr = qt[tau].mean()
    print(f"tau={tau}: mean headroom over forecast {hr:+.3f} pp ≈ {hr / 100 * total_cores:+.0f} vCPUs of {total_cores}")
print("fleet peak 5-min max_cpu (mean of VM maxima): %.1f%% vs avg %.1f%%" % (MX.mean().max(), W.mean().max()))

tau=0.5: mean headroom over forecast -0.128 pp ≈ -36 vCPUs of 28396
tau=0.9: mean headroom over forecast +0.337 pp ≈ +96 vCPUs of 28396
tau=0.95: mean headroom over forecast +0.455 pp ≈ +129 vCPUs of 28396
fleet peak 5-min max_cpu (mean of VM maxima): 20.0% vs avg 9.9%


## 11. Extension: segment-level forecasts
Does the same model rank first when forecasting each VM category separately (backtest only)?

In [19]:
from cwfc.backtest import run_backtest
from cwfc.models import History
rows = []
for k, idx in seg.groupby("vm_category").groups.items():
    pos = np.where(W.index.isin(idx))[0]
    seg_cpu = np.nanmean(hist.X[pos], axis=0)
    h2 = History(hist.timestamps, hist.count, seg_cpu, hist.X[pos], hist.static[pos])
    cands = [mm for mm in models_for("aggregate_cpu_utilization") if mm.name in ("persistence", "moving_average_6", "seasonal_naive_1h", "ses_profile")]
    s2 = run_backtest(h2, cands, "aggregate_cpu_utilization", 24, n_folds=16).summary
    s2["segment"] = k; rows.append(s2[["segment", "model", "mae", "wape_pct", "mae_vs_persistence"]])
segbt = pd.concat(rows).round(4); segbt

,segment,model,mae,wape_pct,mae_vs_persistence
0,Delay-insensitive,ses_profile,0.2667,3.2041,0.9105
1,Delay-insensitive,persistence,0.2929,3.5192,1.0000
2,Delay-insensitive,moving_average_6,0.2997,3.6009,1.0232
3,Delay-insensitive,seasonal_naive_1h,0.3464,4.1617,1.1826
0,Interactive,ses_profile,0.2051,1.7693,0.7400
1,Interactive,seasonal_naive_1h,0.2190,1.8898,0.7904
2,Interactive,moving_average_6,0.2224,1.9191,0.8026
3,Interactive,persistence,0.2771,2.3910,1.0000
0,Unknown,ses_profile,0.3373,3.4268,0.9364
1,Unknown,persistence,0.3602,3.6596,1.0000
